#  Knowledge Graph Extraction Pipeline
**Objectif** : Extraire les entités et relations de documents texte, puis les stocker dans Neo4j.

---
### Étapes :
1.  Connexion au LLM (NVIDIA Mistral Large)
2. Chargement des documents
3. Extraction des entités et relations
4.  Fusion des résultats
5. Génération des requêtes Cypher
6.  Envoi vers Neo4j
Entities + relations in one prompt. Fixed 8000-char chunks.

##  1 — Imports et configuration

In [13]:
import os
import json
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Charger les variables d'environnement (.env)
load_dotenv()

print(f" NVIDIA_API_KEY chargée : {'✓' if os.getenv('NVIDIA_API_KEY') else '✗ MANQUANTE'}")
print(f"  NEO4J_URI : {os.getenv('NEO4J_URI', 'Non défini')}")

 NVIDIA_API_KEY chargée : ✓
  NEO4J_URI : bolt://localhost:7687


##  2 — Initialisation du LLM

In [14]:
llm = ChatNVIDIA(
    model="mistralai/mistral-large-3-675b-instruct-2512",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0,
    max_completion_tokens=32000,
)


## 3 — Test de connexion

In [15]:
response = llm.invoke("Reply with exactly: CONNECTION OK")
print("[Connection test]", response.content.strip())

[Connection test] CONNECTION OK


##  4 — Définition du schéma du graphe

In [ ]:
SCHEMA = """
NODES (entity: fields):
  Company         : name, activity, certifications
  Supplier        : name
  Agreement       : name, description, type, startdate, enddate
  Document        : reference, title, version, date, owner
  Clients         : name, sector
  Clause          : name, description
  governance_body : name, acronyme, role
  Claim           : claim_type, value, unit, scope, source_doc, source_doc_id, quote
  Roles           : title
  Asset           : type, description, classification
  Algorithm       : name, use
  Protocol        : name, use, source
  Risk            : description, level
  Framework       : name, type, use
  Control         : name, requirement, source, Cycle
  Technology      : name, use, source

RELATIONSHIPS (source -[REL]-> target):
  Agreement       -[for]->              Supplier
  Company         -[Use]->              Agreement
  Supplier        -[HAS]->              Company
  Company         -[HAS]->              Document
  Company         -[HAS]->              Clients
  Company         -[HAS]->              governance_body
  Company         -[has_requirement]->  Clause
  Supplier        -[apply_to]->         Clause
  governance_body -[supervise]->        Roles
  governance_body -[approves]->         Claim
  Claim           -[Concerns]->         Asset
  Claim           -[Asserted_by]->      Roles
  Roles           -[operates]->         Technology
  Roles           -[EXECUTES]->         Control
  Algorithm       -[protects]->         Asset
  Protocol        -[used_by]->          Algorithm
  Protocol        -[mitigates]->        Risk
  Risk            -[targeting]->        Asset
  Framework       -[required_by]->      Control
  Control         -[IMPLEMENTED_BY]->   Technology
  Technology      -[Has_access_to]->    Asset
"""

SYSTEM_PROMPT = f"""
You are a knowledge-graph extraction assistant.
Given a text document, extract ALL entities and relationships that match
the schema below. Be exhaustive — extract every entity and relationship you can find.
Return ONLY a valid JSON object, no markdown, no explanation.

{SCHEMA}

IMPORTANT HINTS:
- Any vendor, contractor, or third-party provider → extract as Supplier
- Any contract, MSA, SLA, agreement → extract as Agreement
- Agreement links to Supplier with [for] relation
- Company links to Agreement with [Use] relation
- Supplier links to Clause with [apply_to] relation
- Any threat, attack, or vulnerability targeting an asset → extract Risk -[targeting]-> Asset

Output format:
{{
  "entities": [
    {{"label": "<NodeLabel>", "properties": {{"field": "value"}}}}
  ],
  "relationships": [
    {{
      "from_label": "<NodeLabel>",
      "from_key":   {{"field": "value"}},
      "rel_type":   "<REL_TYPE>",
      "to_label":   "<NodeLabel>",
      "to_key":     {{"field": "value"}}
    }}
  ]
}}

Rules:
- Use ONLY node labels and relationship types from the schema.
- For from_key / to_key use the single most identifying property.
- If a value is unknown, omit the field.
- String values only, no nested objects.
"""
print(" Schéma défini :")
print(f"   - 16 types de nœuds")
print(f"   - 21 types de relations")

✅ Schéma préservé, instructions compactes
   Taille du prompt : 2162 caractères


## 5 — Fonction d'extraction

In [17]:
MAX_CHARS = 8000  # 5 sections au lieu de 44 !

def split_by_section(text: str) -> list:
    """Découpe en sections de taille fixe."""
    sections = []
    start = 0
    while start < len(text):
        end = start + MAX_CHARS
        sections.append(text[start:end])
        start = end
    return sections

def merge_results(results: list) -> dict:
    merged_entities = []
    merged_relations = []
    seen_entities = set()
    seen_relations = set()
    for result in results:
        for entity in result.get("entities", []):
            key = (entity["label"], str(entity.get("properties", {})))
            if key not in seen_entities:
                seen_entities.add(key)
                merged_entities.append(entity)
        for rel in result.get("relationships", []):
            key = (
                rel.get("from_label"),
                str(rel.get("from_key")),
                rel.get("rel_type"),
                rel.get("to_label"),
                str(rel.get("to_key")),
            )
            if key not in seen_relations:
                seen_relations.add(key)
                merged_relations.append(rel)
    return {"entities": merged_entities, "relationships": merged_relations}

def extract_entities(document_text: str) -> dict:
    messages = [
        ("system", SYSTEM_PROMPT),
        ("human", f"Document:\n\n{document_text}"),
    ]
    raw = llm.invoke(messages).content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"      JSON invalide : {e}")
        raise

def extract_full_document(filepath: str) -> dict:
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    print(f"   Taille : {len(text):,} caractères")

    if len(text) <= MAX_CHARS:
        print(f"   Document court → 1 seul appel")
        return extract_entities(text)

    print(f"   Document long → découpage par section")
    sections = split_by_section(text)
    print(f"   {len(sections)} sections trouvées")

    all_results = []
    for i, section in enumerate(sections):
        print(f"      → Section {i+1}/{len(sections)} ({len(section):,} chars)...", end=" ", flush=True)
        try:
            result = extract_entities(section)
            all_results.append(result)
            print(f"✅ {len(result.get('entities', []))} entités")
        except Exception as e:
            print(f"❌ {e}")
            all_results.append({"entities": [], "relationships": []})

    return merge_results(all_results)

print("✅ Fonctions prêtes")

✅ Fonctions prêtes


##  6 — Chargement des documents

In [18]:
DOCUMENTS = [
    "/home/maroua/Desktop/cassiope/SecuraOps High.txt",
    "/home/maroua/Desktop/cassiope/SafeLink Contradictory.txt",
]

documents_text = {}
for filepath in DOCUMENTS:
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    documents_text[filepath] = text
    print(f" {filepath} : {len(text):,} caractères chargés")

 /home/maroua/Desktop/cassiope/SecuraOps High.txt : 39,933 caractères chargés
 /home/maroua/Desktop/cassiope/SafeLink Contradictory.txt : 21,896 caractères chargés


## 7 — Extraction des entités (document par document)

In [19]:
raw_results = {}

for filepath in DOCUMENTS:
    nom = filepath.split("/")[-1].replace(".txt", "")
    print(f"\n Extraction : {nom}")
    result = extract_full_document(filepath)  
    raw_results[filepath] = result

    with open(f"{nom}_graph.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f"   Total : {len(result['entities'])} entités, {len(result['relationships'])} relations")
    print(f"   Sauvegardé : {nom}_graph.json")


 Extraction : SecuraOps High
   Taille : 39,933 caractères
   Document long → découpage par section
   5 sections trouvées
      → Section 1/5 (8,000 chars)... ❌ [504] Gateway Timeout
{'_content': b'', '_content_consumed': True, '_next': None, 'status_code': 504, 'headers': {'Date': 'Wed, 11 Mar 2026 19:06:51 GMT', 'Content-Length': '0', 'Connection': 'keep-alive', 'Access-Control-Expose-Headers': 'nvcf-reqid', 'Nvcf-Reqid': '357d6636-470a-479e-b691-f7f086e431ec', 'Nvcf-Status': 'errored', 'Vary': 'Origin'}, 'raw': <urllib3.response.HTTPResponse object at 0x760b49d41990>, 'url': 'https://integrate.api.nvidia.com/v1/chat/completions', 'encoding': None, 'history': [], 'reason': 'Gateway Timeout', 'cookies': <RequestsCookieJar[]>, 'elapsed': datetime.timedelta(seconds=302, microseconds=386021), 'request': <PreparedRequest [POST]>, 'connection': <requests.adapters.HTTPAdapter object at 0x760b490ea1d0>}
      → Section 2/5 (8,000 chars)... 

KeyboardInterrupt: 

##  8 — Fusion des résultats (sans doublons)

## 9 — Visualisation du JSON extrait

In [ ]:
# Aperçu des résultats pour chaque document
for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1]
    print(f"\n{nom}")
    print(f"   Entités   : {len(result['entities'])}")
    print(f"   Relations : {len(result['relationships'])}")
    
    # Résumé par type de nœud
    from collections import Counter
    node_counts = Counter(e['label'] for e in result['entities'])
    print("   Types d'entités :")
    for label, count in sorted(node_counts.items()):
        print(f"      {label:<20} : {count}")


SecuraOps High.txt
   Entités   : 197
   Relations : 185
   Types d'entités :
      Agreement            : 1
      Algorithm            : 5
      Asset                : 20
      Claim                : 13
      Clause               : 2
      Clients              : 1
      Company              : 1
      Control              : 19
      Document             : 22
      Framework            : 13
      Protocol             : 3
      Risk                 : 10
      Roles                : 36
      Supplier             : 15
      Technology           : 29
      governance_body      : 7

SafeLink Contradictory.txt
   Entités   : 111
   Relations : 104
   Types d'entités :
      Agreement            : 10
      Algorithm            : 1
      Asset                : 6
      Claim                : 6
      Clause               : 18
      Clients              : 1
      Company              : 1
      Control              : 17
      Document             : 12
      Framework            : 2
      Protocol 

##  10 — Sauvegarde du JSON

In [ ]:
# Bloc 10 — Sauvegarde JSON pour chaque document
for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1].replace(".txt", "")
    output_file = f"{nom}_graph.json"
    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    
    print(f" Sauvegardé : {output_file}")

 Sauvegardé : SecuraOps High_graph.json
 Sauvegardé : SafeLink Contradictory_graph.json


##  11 — Génération des requêtes Cypher

In [20]:
def json_to_cypher(extracted: dict, company_label: str = None) -> list:
    statements = []
    
    for entity in extracted.get("entities", []):
        label = entity["label"]
        props = entity.get("properties", {})
        if props:
            # Ajouter le label entreprise si fourni
            if company_label:
                statements.append(f"MERGE (n:{label}:{company_label} {props_str(props)});")
            else:
                statements.append(f"MERGE (n:{label} {props_str(props)});")

    for rel in extracted.get("relationships", []):
        fl = rel["from_label"]
        fk = props_str(rel["from_key"])
        rt = rel["rel_type"]
        tl = rel["to_label"]
        tk = props_str(rel["to_key"])
        statements.append(
            f"MATCH (a:{fl} {fk}), (b:{tl} {tk}) MERGE (a)-[:{rt}]->(b);"
        )
    return statements

# Mapper fichier → nom entreprise
company_map = {
    "SecuraOps High": "SecuraOps",
    "SafeLink Contradictory": "SafeLink",
}

for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1].replace(".txt", "")
    company = company_map.get(nom, nom)
    stmts = json_to_cypher(result, company_label=company)
    print(f"{nom} : {len(stmts)} statements")

In [21]:
# Sauvegarde des requêtes Cypher pour chaque entreprise
for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1].replace(".txt", "")
    company = company_map.get(nom, nom)
    stmts = json_to_cypher_with_label(result, company)
    
    output_file = f"{nom}_cypher.cypher"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n".join(stmts))
    
    print(f"💾 Sauvegardé : {output_file} ({len(stmts)} statements)")

## 13 — Envoi vers Neo4j

In [24]:
from neo4j import GraphDatabase

# D'abord vider Neo4j pour repartir propre
uri      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
user     = os.getenv("NEO4J_USER",     "neo4j")
password = os.getenv("NEO4J_PASSWORD", "")

driver = GraphDatabase.driver(uri, auth=(user, password))

# Étape 1 — Vider la base
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print(" Base vidée")

# Mapper fichier → nom entreprise
company_map = {
    "SecuraOps High": "SecuraOps",
    "SafeLink Contradictory": "SafeLink",
}

# Étape 2 — Réenvoyer avec labels séparés
def json_to_cypher_with_label(extracted: dict, company_label: str) -> list:
    statements = []
    for entity in extracted.get("entities", []):
        label = entity["label"]
        props = entity.get("properties", {})
        if props:
            statements.append(f"MERGE (n:{label}:{company_label} {props_str(props)});")
    for rel in extracted.get("relationships", []):
        fl = rel["from_label"]
        fk = props_str(rel["from_key"])
        rt = rel["rel_type"]
        tl = rel["to_label"]
        tk = props_str(rel["to_key"])
        statements.append(
            f"MATCH (a:{fl}:{company_label} {fk}), (b:{tl}:{company_label} {tk}) "
            f"MERGE (a)-[:{rt}]->(b);"
        )
    return statements

for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1].replace(".txt", "")
    company = company_map.get(nom, nom)
    stmts = json_to_cypher_with_label(result, company)
    
    print(f"\n Envoi de {company} vers Neo4j...")
    errors = 0
    with driver.session() as session:
        for stmt in stmts:
            try:
                session.run(stmt)
            except Exception as e:
                errors += 1
    
    print(f"   {len(stmts) - errors} statements OK")
    if errors:
        print(f"    {errors} erreurs")

driver.close()
print("\n Terminé !")

 Base vidée

 Terminé !


In [ ]:
from neo4j import GraphDatabase

# D'abord vider Neo4j pour repartir propre
uri      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
user     = os.getenv("NEO4J_USER",     "neo4j")
password = os.getenv("NEO4J_PASSWORD", "")

driver = GraphDatabase.driver(uri, auth=(user, password))

In [27]:
import json
import os
from dotenv import load_dotenv

load_dotenv()

# Charger les JSON déjà extraits
raw_results = {}

files = {
    "/home/maroua/Desktop/cassiope/SecuraOps High.txt": "/home/maroua/Desktop/cassiope/SecuraOps High_graph.json",
    "/home/maroua/Desktop/cassiope/SafeLink Contradictory.txt": "/home/maroua/Desktop/cassiope/SafeLink Contradictory_graph.json",
}

for filepath, json_file in files.items():
    with open(json_file, "r", encoding="utf-8") as f:
        raw_results[filepath] = json.load(f)
    print(f" Chargé : {json_file.split('/')[-1]} — {len(raw_results[filepath]['entities'])} entités")

 Chargé : SecuraOps High_graph.json — 484 entités
 Chargé : SafeLink Contradictory_graph.json — 111 entités


In [28]:
# ── Sauvegarde Cypher ──────────────────────────────────────────────
for filepath, result in raw_results.items():
    nom     = filepath.split("/")[-1].replace(".txt", "")
    company = company_map.get(nom, nom)
    stmts   = json_to_cypher_with_label(result, company)
    
    output_file = f"/home/maroua/Desktop/cassiope/{nom}_cypher.cypher"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n".join(stmts))
    
    print(f"{nom}_cypher.cypher — {len(stmts)} statements")

SecuraOps High_cypher.cypher — 1011 statements
SafeLink Contradictory_cypher.cypher — 215 statements


In [29]:
from neo4j import GraphDatabase

def props_str(props: dict) -> str:
    parts = []
    for k, v in props.items():
        safe_v = str(v).replace("'", "\\'")
        parts.append(f"{k}: '{safe_v}'")
    return "{" + ", ".join(parts) + "}"

def json_to_cypher_with_label(extracted: dict, company_label: str) -> list:
    statements = []
    for entity in extracted.get("entities", []):
        label = entity["label"]
        props = entity.get("properties", {})
        if props:
            statements.append(f"MERGE (n:{label}:{company_label} {props_str(props)});")
    for rel in extracted.get("relationships", []):
        fl = rel["from_label"]
        fk = props_str(rel["from_key"])
        rt = rel["rel_type"]
        tl = rel["to_label"]
        tk = props_str(rel["to_key"])
        statements.append(
            f"MATCH (a:{fl}:{company_label} {fk}), (b:{tl}:{company_label} {tk}) "
            f"MERGE (a)-[:{rt}]->(b);"
        )
    return statements

company_map = {
    "SecuraOps High": "SecuraOps",
    "SafeLink Contradictory": "SafeLink",
}

uri      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
user     = os.getenv("NEO4J_USER",     "neo4j")
password = os.getenv("NEO4J_PASSWORD", "")

driver = GraphDatabase.driver(uri, auth=(user, password))

# Vider la base
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("🗑️ Base vidée")

# Envoyer chaque entreprise
for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1].replace(".txt", "")
    company = company_map.get(nom, nom)
    stmts = json_to_cypher_with_label(result, company)

    print(f"\n📤 Envoi de {company}...")
    errors = 0
    with driver.session() as session:
        for stmt in stmts:
            try:
                session.run(stmt)
            except Exception:
                errors += 1

    print(f"   {len(stmts) - errors} statements OK")
    if errors:
        print(f"    {errors} erreurs")

driver.close()
print("\nTerminé ! → http://localhost:7474")

🗑️ Base vidée

📤 Envoi de SecuraOps...
   1011 statements OK

📤 Envoi de SafeLink...
   215 statements OK

Terminé ! → http://localhost:7474
